<a href="https://colab.research.google.com/github/Dorisa1985/mining-plan-vs-actual-viz-synthetic/blob/main/mining_plan_vs_actual_viz_synthetic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import altair as alt

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/Dorisa1985/mining-plan-vs-actual-viz-synthetic/main/mining_synthetic.csv")


In [ ]:
df.head(10)

,date,plan_fleet_hours,plan_cycle_min,plan_payload_bcm,plan_volume_bcm,loss_rain_hr,loss_truck_breakdown_hr,loss_digger_breakdown_hr,loss_standby_hr,loss_fueling_shiftchange_hr,actual_fleet_hours,actual_queue_min,actual_other_cycle_extra_min,actual_cycle_min,actual_payload_bcm,actual_volume_bcm
0,2026-09-01,400.0,30.0,40.0,32000.0,0.0,3.83,0.00,2.04,5.88,388.25,0.78,0.13,30.91,40.10,30220.95
1,2026-09-02,400.0,30.0,40.0,32000.0,0.0,7.96,7.05,1.99,2.06,380.94,0.39,0.30,30.69,39.92,29730.45
2,2026-09-03,400.0,30.0,40.0,32000.0,0.0,0.00,5.50,2.10,2.29,390.11,0.33,0.33,30.66,40.07,30590.43
3,2026-09-04,400.0,30.0,40.0,32000.0,0.0,12.58,6.41,0.71,3.08,377.22,0.76,0.18,30.94,39.45,28858.43
4,2026-09-05,400.0,30.0,40.0,32000.0,70.0,9.55,0.34,2.26,5.18,312.67,0.68,3.02,33.70,39.96,22245.03
5,2026-09-06,400.0,30.0,40.0,32000.0,115.0,5.83,5.05,8.88,8.05,257.19,0.85,3.56,34.41,40.09,17978.63
6,2026-09-07,400.0,30.0,40.0,32000.0,60.0,5.75,1.44,3.22,4.43,325.16,0.55,2.26,32.81,39.96,23761.16
7,2026-09-08,400.0,30.0,40.0,32000.0,0.0,8.22,1.63,5.77,5.38,379.00,0.91,1.50,32.41,39.63,27805.81
8,2026-09-09,400.0,30.0,40.0,32000.0,0.0,5.93,4.52,1.35,3.41,384.79,0.43,0.45,30.88,38.39,28702.24
9,2026-09-10,340.0,30.0,40.0,27200.0,0.0,6.10,5.63,5.51,3.43,319.33,0.23,0.50,30.73,39.90,24877.19


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   date                          30 non-null     object 
 1   plan_fleet_hours              30 non-null     float64
 2   plan_cycle_min                30 non-null     float64
 3   plan_payload_bcm              30 non-null     float64
 4   plan_volume_bcm               30 non-null     float64
 5   loss_rain_hr                  30 non-null     float64
 6   loss_truck_breakdown_hr       30 non-null     float64
 7   loss_digger_breakdown_hr      30 non-null     float64
 8   loss_standby_hr               30 non-null     float64
 9   loss_fueling_shiftchange_hr   30 non-null     float64
 10  actual_fleet_hours            30 non-null     float64
 11  actual_queue_min              30 non-null     float64
 12  actual_other_cycle_extra_min  30 non-null     float64
 13  actual_

In [ ]:
# unit conversion from minutes to hours
df["ct_plan_hr"]=df["plan_cycle_min"]/60
df["ct_act_hr"]=df["actual_cycle_min"]/60

In [ ]:
# create 3 new columns
df["eff_hours"]=(df["actual_fleet_hours"]-df["plan_fleet_hours"])*df["plan_payload_bcm"]/df["ct_plan_hr"]
df["ct_eff"]=df["actual_fleet_hours"]*df["plan_payload_bcm"]*((1/df["ct_act_hr"])-(1/df["ct_plan_hr"]))
df["eff_pyld"]=df["actual_fleet_hours"]*(df["actual_payload_bcm"]-df["plan_payload_bcm"])/df["ct_act_hr"]

In [ ]:
df.head(6)

,date,plan_fleet_hours,plan_cycle_min,plan_payload_bcm,plan_volume_bcm,loss_rain_hr,loss_truck_breakdown_hr,loss_digger_breakdown_hr,loss_standby_hr,loss_fueling_shiftchange_hr,...,actual_queue_min,actual_other_cycle_extra_min,actual_cycle_min,actual_payload_bcm,actual_volume_bcm,ct_plan_hr,ct_act_hr,eff_hours,ct_eff,eff_pyld
0,2026-09-01,400.0,30.0,40.0,32000.0,0.0,3.83,0.00,2.04,5.88,...,0.78,0.13,30.91,40.10,30220.95,0.5,0.515167,-940.0,-914.416047,75.363960
1,2026-09-02,400.0,30.0,40.0,32000.0,0.0,7.96,7.05,1.99,2.06,...,0.39,0.30,30.69,39.92,29730.45,0.5,0.511500,-1524.8,-685.170674,-59.580059
2,2026-09-03,400.0,30.0,40.0,32000.0,0.0,0.00,5.50,2.10,2.29,...,0.33,0.33,30.66,40.07,30590.43,0.5,0.511000,-791.2,-671.813699,53.439726
3,2026-09-04,400.0,30.0,40.0,32000.0,0.0,12.58,6.41,0.71,3.08,...,0.76,0.18,30.94,39.45,28858.43,0.5,0.515667,-1822.4,-916.837233,-402.335488
4,2026-09-05,400.0,30.0,40.0,32000.0,70.0,9.55,0.34,2.26,5.18,...,0.68,3.02,33.70,39.96,22245.03,0.5,0.561667,-6986.4,-2746.300297,-22.267300
5,2026-09-06,400.0,30.0,40.0,32000.0,115.0,5.83,5.05,8.88,8.05,...,0.85,3.56,34.41,40.09,17978.63,0.5,0.573500,-11424.8,-2636.926242,40.361116


In [ ]:
gap=df["actual_volume_bcm"]-df["plan_volume_bcm"]
print((df[["eff_hours","ct_eff","eff_pyld"]].sum(axis=1)-gap).abs().max())
print(gap.sum())

0.0048735832642705645
-156916.84


In [ ]:
df["total_loss_hr"]=df[["loss_rain_hr","loss_truck_breakdown_hr","loss_digger_breakdown_hr","loss_standby_hr","loss_fueling_shiftchange_hr"]].sum(axis=1)

In [ ]:
df["eff_rain"]=(df["loss_rain_hr"]/df["total_loss_hr"])*df["eff_hours"]
df["eff_truck_bd"]=(df["loss_truck_breakdown_hr"]/df["total_loss_hr"])*df["eff_hours"]
df["eff_digger_bd"]=(df["loss_digger_breakdown_hr"]/df["total_loss_hr"])*df["eff_hours"]
df["eff_standby"]=(df["loss_standby_hr"]/df["total_loss_hr"])*df["eff_hours"]
df["eff_fuel-shift"]=(df["loss_fueling_shiftchange_hr"]/df["total_loss_hr"])*df["eff_hours"]

In [ ]:
df["eff_queue"]=df["actual_queue_min"]*df["ct_eff"]/(df["actual_queue_min"]+df["actual_other_cycle_extra_min"])
df["eff_other_cycle"]=df["actual_other_cycle_extra_min"]*df["ct_eff"]/(df["actual_queue_min"]+df["actual_other_cycle_extra_min"])

In [ ]:
effect_cols=["eff_rain","eff_truck_bd","eff_digger_bd","eff_standby","eff_fuel-shift","eff_queue","eff_other_cycle","eff_pyld"]

# wide to long
long=df.melt(id_vars="date",value_vars=effect_cols,var_name="factor",value_name="effect_bcm")


In [ ]:
wf = long.groupby("factor",as_index=False)["effect_bcm"].sum().sort_values("effect_bcm").reset_index(drop=True)

In [ ]:
wf["end"]=wf["effect_bcm"].cumsum()
wf["start"]=wf["end"]-wf["effect_bcm"]

In [ ]:
wf

,factor,effect_bcm,end,start
0,eff_queue,-24000.001583,-24000.001583,0.000000
1,eff_rain,-23040.000000,-47040.001583,-24000.001583
2,eff_digger_bd,-21296.000000,-68336.001583,-47040.001583
3,eff_pyld,-20655.592853,-88991.594437,-68336.001583
4,eff_truck_bd,-20416.800000,-109408.394437,-88991.594437
5,eff_standby,-18374.400000,-127782.794437,-109408.394437
6,eff_other_cycle,-17506.053841,-145288.848277,-127782.794437
7,eff_fuel-shift,-11628.000000,-156916.848277,-145288.848277


In [ ]:
total=wf["effect_bcm"].sum()


In [ ]:
wf=pd.concat([wf, pd.DataFrame({"factor":["total_gap"],
                                "effect_bcm":[total],
                                "start":[0],
                                "end":[total]})],
             ignore_index=True)

In [ ]:
wf

,factor,effect_bcm,end,start
0,eff_queue,-24000.001583,-24000.001583,0.000000
1,eff_rain,-23040.000000,-47040.001583,-24000.001583
2,eff_digger_bd,-21296.000000,-68336.001583,-47040.001583
3,eff_pyld,-20655.592853,-88991.594437,-68336.001583
4,eff_truck_bd,-20416.800000,-109408.394437,-88991.594437
5,eff_standby,-18374.400000,-127782.794437,-109408.394437
6,eff_other_cycle,-17506.053841,-145288.848277,-127782.794437
7,eff_fuel-shift,-11628.000000,-156916.848277,-145288.848277
8,total_gap,-156916.848277,-156916.848277,0.000000


In [ ]:
wf=wf[["factor","effect_bcm","start","end"]]

In [ ]:
wf

,factor,effect_bcm,start,end
0,eff_queue,-24000.001583,0.000000,-24000.001583
1,eff_rain,-23040.000000,-24000.001583,-47040.001583
2,eff_digger_bd,-21296.000000,-47040.001583,-68336.001583
3,eff_pyld,-20655.592853,-68336.001583,-88991.594437
4,eff_truck_bd,-20416.800000,-88991.594437,-109408.394437
5,eff_standby,-18374.400000,-109408.394437,-127782.794437
6,eff_other_cycle,-17506.053841,-127782.794437,-145288.848277
7,eff_fuel-shift,-11628.000000,-145288.848277,-156916.848277
8,total_gap,-156916.848277,0.000000,-156916.848277


In [ ]:
labels = {"eff_rain": "Rain / wet roads", "eff_truck_bd": "Truck breakdown",
          "eff_digger_bd": "Digger breakdown", "eff_standby": "Standby",
          "eff_fuel-shift": "Fueling & shift change", "eff_queue": "Loading queue",
          "eff_other_cycle": "Other cycle delays", "eff_pyld": "Payload",
          "total_gap": "TOTAL SHORTFALL"}

In [ ]:
wf["label"]=wf["factor"].map(labels)

/tmp/ipykernel_3212/2847544371.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wf["label"]=wf["factor"].map(labels)


In [ ]:
wf

,factor,effect_bcm,start,end,label
0,eff_queue,-24000.001583,0.000000,-24000.001583,Loading queue
1,eff_rain,-23040.000000,-24000.001583,-47040.001583,Rain / wet roads
2,eff_digger_bd,-21296.000000,-47040.001583,-68336.001583,Digger breakdown
3,eff_pyld,-20655.592853,-68336.001583,-88991.594437,Payload
4,eff_truck_bd,-20416.800000,-88991.594437,-109408.394437,Truck breakdown
5,eff_standby,-18374.400000,-109408.394437,-127782.794437,Standby
6,eff_other_cycle,-17506.053841,-127782.794437,-145288.848277,Other cycle delays
7,eff_fuel-shift,-11628.000000,-145288.848277,-156916.848277,Fueling & shift change
8,total_gap,-156916.848277,0.000000,-156916.848277,TOTAL SHORTFALL


In [ ]:
wf["type"]=wf["factor"].apply(lambda f:"Total" if f == "total_gap" else "Factor")

/tmp/ipykernel_3212/1687171951.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wf["type"]=wf["factor"].apply(lambda f:"Total" if f == "total_gap" else "Factor")


In [ ]:
wf

,factor,effect_bcm,start,end,label,type
0,eff_queue,-24000.001583,0.000000,-24000.001583,Loading queue,Factor
1,eff_rain,-23040.000000,-24000.001583,-47040.001583,Rain / wet roads,Factor
2,eff_digger_bd,-21296.000000,-47040.001583,-68336.001583,Digger breakdown,Factor
3,eff_pyld,-20655.592853,-68336.001583,-88991.594437,Payload,Factor
4,eff_truck_bd,-20416.800000,-88991.594437,-109408.394437,Truck breakdown,Factor
5,eff_standby,-18374.400000,-109408.394437,-127782.794437,Standby,Factor
6,eff_other_cycle,-17506.053841,-127782.794437,-145288.848277,Other cycle delays,Factor
7,eff_fuel-shift,-11628.000000,-145288.848277,-156916.848277,Fueling & shift change,Factor
8,total_gap,-156916.848277,0.000000,-156916.848277,TOTAL SHORTFALL,Total


In [ ]:
group = {"eff_rain": "Lost hours", "eff_truck_bd": "Lost hours", "eff_digger_bd": "Lost hours",
         "eff_standby": "Lost hours", "eff_fuel-shift": "Lost hours",
         "eff_queue": "Cycle time delays", "eff_other_cycle": "Cycle time delays",
         "eff_pyld": "Payload shortfall", "total_gap": "Total shortfall"}



wf["group"] = wf["factor"].map(group)

wf[["factor", "label", "group", "effect_bcm", "start", "end"]]

/tmp/ipykernel_3212/823820029.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wf["group"] = wf["factor"].map(group)


,factor,label,group,effect_bcm,start,end
0,eff_queue,Loading queue,Cycle time delays,-24000.001583,0.000000,-24000.001583
1,eff_rain,Rain / wet roads,Lost hours,-23040.000000,-24000.001583,-47040.001583
2,eff_digger_bd,Digger breakdown,Lost hours,-21296.000000,-47040.001583,-68336.001583
3,eff_pyld,Payload,Payload shortfall,-20655.592853,-68336.001583,-88991.594437
4,eff_truck_bd,Truck breakdown,Lost hours,-20416.800000,-88991.594437,-109408.394437
5,eff_standby,Standby,Lost hours,-18374.400000,-109408.394437,-127782.794437
6,eff_other_cycle,Other cycle delays,Cycle time delays,-17506.053841,-127782.794437,-145288.848277
7,eff_fuel-shift,Fueling & shift change,Lost hours,-11628.000000,-145288.848277,-156916.848277
8,total_gap,TOTAL SHORTFALL,Total shortfall,-156916.848277,0.000000,-156916.848277


In [ ]:
order= list(wf["label"])
order

['Loading queue',
 'Rain / wet roads',
 'Digger breakdown',
 'Payload',
 'Truck breakdown',
 'Standby',
 'Other cycle delays',
 'Fueling & shift change',
 'TOTAL SHORTFALL']

In [ ]:
color = alt.Scale(domain=["Lost hours", "Cycle time delays", "Payload shortfall", "Total shortfall"],
                  range=["#c25e00", "#7b4f9e", "#2fa38f", "#333a45"])
x = alt.X("label:N", sort=list(wf["label"]), title=None, axis=alt.Axis(labelAngle=-30))

bars = alt.Chart(wf).mark_bar().encode(
    x=x,
    y=alt.Y("start:Q", title="Contribution to the total shortfall (BCM)",
            scale=alt.Scale(domain=[-172000, 0], nice=False)),
    y2="end:Q",
    color=alt.Color("group:N", scale=color, title=None),
    tooltip=[alt.Tooltip("label:N", title="Factor"),
             alt.Tooltip("group:N", title="Group"),
             alt.Tooltip("effect_bcm:Q", title="Effect (BCM)", format=",.0f")])

number= alt.Chart(wf).mark_text(dy=14, fontWeight="bold").encode(
    x=x, y="end:Q", text=alt.Text("effect_bcm:Q", format=",.0f"))

chart = (bars + number).properties(
    width=640, height=380,
    title=alt.TitleParams("Where the 156,917 BCM total shortfall came from",
                          subtitle="Lost hours 60%  ·  Cycle time delays 27%  ·  Payload shortfall 13%",
                          fontSize=16, anchor="middle"))
chart

alt.LayerChart(...)

In [ ]:
plan   = df["plan_volume_bcm"].sum()
actual = df["actual_volume_bcm"].sum()

m = pd.DataFrame({
    "bar":   ["Plan", "Actual", "Actual"],
    "part":  ["Plan", "Actual", "Total shortfall"],
    "start": [0, 0, actual],
    "end":   [plan, actual, plan]})
m["label"] = (m["end"] - m["start"]).map("{:,.0f}".format)
m.loc[m["part"] == "Total shortfall", "label"] = "−" + m.loc[m["part"] == "Total shortfall", "label"]
m["mid"]   = (m["start"] + m["end"]) / 2

color_m = alt.Scale(domain=["Plan", "Actual", "Total shortfall"],
                    range=["#6b7280", "#2f5d8a", "#333a45"])
x_m = alt.X("bar:N", sort=["Plan", "Actual"], title=None,
            axis=alt.Axis(labelAngle=0, labelFontSize=13))

bars_m = alt.Chart(m).mark_bar(size=90).encode(
    x=x_m,
    y=alt.Y("start:Q", title="Monthly hauled volume (BCM)",
            scale=alt.Scale(domain=[0, 1100000]), axis=alt.Axis(format=",.0f")),
    y2="end:Q",
    color=alt.Color("part:N", scale=color_m, title=None),
    tooltip=[alt.Tooltip("part:N", title="Part"), alt.Tooltip("label:N", title="Volume (BCM)")])

number_m = alt.Chart(m).mark_text(color="white", fontWeight="bold", fontSize=13).encode(
    x=x_m, y="mid:Q", text="label:N")

chart1 = (bars_m + number_m).properties(
    width=420, height=360,
    title=alt.TitleParams("September 2026: actual reached %.1f%% of planned volume"
                      % (100 * actual / plan),
                      subtitle="Synthetic data for illustration",
                      subtitleColor="#6b7280", fontSize=16, anchor="middle"))
chart1

alt.LayerChart(...)

In [ ]:
color = alt.Scale(domain=["Lost hours", "Cycle time delays", "Payload shortfall", "Total shortfall"],
                  range=["#c25e00", "#7b4f9e", "#2fa38f", "#333a45"])
x = alt.X("label:N", sort=list(wf["label"]), title="Contributing factor", axis=alt.Axis(labelAngle=-30))

bars = alt.Chart(wf).mark_bar().encode(
    x=x,
    y=alt.Y("start:Q", title="Contribution to the total shortfall (BCM)",
            scale=alt.Scale(domain=[-172000, 0], nice=False)),
    y2="end:Q",
    color=alt.Color("group:N", scale=color, title=None),
    tooltip=[alt.Tooltip("label:N", title="Factor"),
             alt.Tooltip("group:N", title="Group"),
             alt.Tooltip("effect_bcm:Q", title="Effect (BCM)", format=",.0f")])

number= alt.Chart(wf).mark_text(dy=14, fontWeight="bold").encode(
    x=x, y="end:Q", text=alt.Text("effect_bcm:Q", format=",.0f"))

chart = (bars + number).properties(
    width=640, height=380,
    title=alt.TitleParams("Where the 156,917 BCM total shortfall came from",
                          subtitle="Lost hours 60%  ·  Cycle time delays 27%  ·  Payload shortfall 13%",
                          fontSize=16, anchor="middle"))
chart

alt.LayerChart(...)

In [ ]:
print(set(wf["group"]) - {"Lost hours", "Cycle time delays", "Payload shortfall", "Total shortfall"})
print(wf[["start", "end"]].isna().sum())
wf[["label", "group", "effect_bcm", "start", "end"]]

set()
start    0
end      0
dtype: int64


,label,group,effect_bcm,start,end
0,Loading queue,Cycle time delays,-24000.001583,0.000000,-24000.001583
1,Rain / wet roads,Lost hours,-23040.000000,-24000.001583,-47040.001583
2,Digger breakdown,Lost hours,-21296.000000,-47040.001583,-68336.001583
3,Payload,Payload shortfall,-20655.592853,-68336.001583,-88991.594437
4,Truck breakdown,Lost hours,-20416.800000,-88991.594437,-109408.394437
5,Standby,Lost hours,-18374.400000,-109408.394437,-127782.794437
6,Other cycle delays,Cycle time delays,-17506.053841,-127782.794437,-145288.848277
7,Fueling & shift change,Lost hours,-11628.000000,-145288.848277,-156916.848277
8,TOTAL SHORTFALL,Total shortfall,-156916.848277,0.000000,-156916.848277


In [ ]:
plan   = df["plan_volume_bcm"].sum()
actual = df["actual_volume_bcm"].sum()

m = pd.DataFrame({
    "bar":   ["Plan", "Actual", "Actual"],
    "part":  ["Plan", "Actual", "Total shortfall"],
    "start": [0, 0, actual],
    "end":   [plan, actual, plan]})
m["label"] = (m["end"] - m["start"]).map("{:,.0f}".format)
m.loc[m["part"] == "Total shortfall", "label"] = "Shortfall\n−" + m.loc[m["part"] == "Total shortfall", "label"]
m["mid"]   = (m["start"] + m["end"]) / 2

color_m = alt.Scale(domain=["Plan", "Actual", "Total shortfall"],
                    range=["#6b7280", "#2f5d8a", "#333a45"])
x_m = alt.X("bar:N", sort=["Plan", "Actual"], title=None,
            axis=alt.Axis(labelAngle=0, labelFontSize=13))

bars_m = alt.Chart(m).mark_bar(size=90).encode(
    x=x_m,
    y=alt.Y("start:Q", title="Monthly hauled volume (BCM)",
            scale=alt.Scale(domain=[0, 1100000]), axis=alt.Axis(format=",.0f")),
    y2="end:Q",
    color=alt.Color("part:N", scale=color_m, title=None),
    tooltip=[alt.Tooltip("part:N", title="Part"), alt.Tooltip("label:N", title="Volume (BCM)")])

number_m = alt.Chart(m).mark_text(color="white", fontWeight="bold", fontSize=13, lineBreak="\n").encode(
    x=x_m, y="mid:Q", text="label:N")

chart1 = (bars_m + number_m).properties(
    width=420, height=360,
    title=alt.TitleParams("September 2026: actual reached %.1f%% of planned volume"
                      % (100 * actual / plan),
                      subtitle="Synthetic data for illustration",
                      subtitleColor="#6b7280", fontSize=16, anchor="middle"))
chart1

alt.LayerChart(...)